# IZQAR PaySim v2 compatibility run
Add Input: `ealaxi/paysim1`. This reproduces the locked PaySim training configuration with scikit-learn 1.9.1, matching the local IEEE-CIS API runtime. It writes new v2 files and does not overwrite the original v1 result. The same previously inspected holdout is used only for a compatibility reproduction; its metrics are not an independent new validation of model improvements.


In [ ]:
from pathlib import Path
import subprocess, sys
root = Path('/kaggle/working/izqar_v2')
if root.exists():
    raise FileExistsError('izqar_v2 already exists; use a fresh notebook session to avoid stale code')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feat/ml-pipelines-and-startup', 'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(root / 'backend/requirements-ml.txt'), 'scikit-learn==1.9.1'], check=True)
matches = list(Path('/kaggle/input').rglob('PS_20174392719_1491204439457_log.csv'))
if len(matches) != 1:
    raise FileNotFoundError(f'Expected one PaySim CSV; found {len(matches)}. Use Add Input: ealaxi/paysim1.')
print('PaySim input:', matches[0], 'bytes:', matches[0].stat().st_size)


In [ ]:
import json, zipfile
backend = root / 'backend'
output = Path('/kaggle/working/paysim_model_v2.joblib')
subprocess.run([sys.executable, str(backend / 'scripts/train_paysim.py'), '--data', str(matches[0]), '--output', str(output), '--estimators', '180', '--full-data', '--model-version', 'paysim-xgboost-v2'], cwd=backend, check=True)
report = output.with_suffix('.metrics.json')
metadata = json.loads(report.read_text(encoding='utf-8'))
assert metadata['library_versions']['scikit_learn'] == '1.9.1'
assert metadata['model_version'] == 'paysim-xgboost-v2'
bundle = Path('/kaggle/working/izqar_paysim_artifacts_v2.zip')
if bundle.exists():
    raise FileExistsError(f'Refusing to overwrite {bundle}')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(output, output.name)
    archive.write(report, report.name)
print('Download from Kaggle Output:', bundle)


In [ ]:
audit = Path('/kaggle/working/paysim_model_v2.validation_audit.json')
subprocess.run([sys.executable, str(backend / 'scripts/audit_paysim_validation.py'), '--data', str(matches[0]), '--model', str(output), '--output', str(audit)], cwd=backend, check=True)
with zipfile.ZipFile(bundle, 'a', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(audit, audit.name)
print('Validation-only robustness audit added:', audit)
